# Packaging & Tooling Notes — p2p-portfolio

A working record of the packaging/tooling topics covered recently: installing the project, running its test suite, what pytest actually does, how to confirm an editable install, how TOML is written, and where `pip show` gets its information from.

Run this notebook from inside the `p2p-portfolio/` project root, after extracting the repo, so the relative paths resolve correctly.

## 1. Installing the project

`pip install -e ".[dev]"` installs the package in **editable mode** plus the `dev` extras (pytest, pytest-cov, ruff).

- `-e` (editable): instead of copying your files into `site-packages`, pip links straight back to your `src/p2p_portfolio/` folder. Edit the source, rerun your code — no reinstall needed.
- `.`: install whatever package is defined by the `pyproject.toml` in the current directory.
- `[dev]`: pulls in the optional dependency group named `dev`, defined in `pyproject.toml` under `[project.optional-dependencies]`.

Run once per environment:
```bash
pip install -e ".[dev]"
```

In [1]:
# If you haven't installed yet, uncomment and run:
# !pip install -e ".[dev]" --break-system-packages -q
print("Run the install command above from a terminal the first time you set this project up.")

Run the install command above from a terminal the first time you set this project up.


## 2. Running the test suite

```bash
pytest tests/ -v
```

- `tests/` — the folder to search (matches `testpaths = ["tests"]` in `pyproject.toml`)
- `-v` — verbose: prints each test's name and PASSED/FAILED, instead of just dots

Useful variations:
```bash
pytest tests/test_metrics_spread.py -v                               # one file
pytest tests/test_metrics_spread.py::test_captured_spread_pct -v     # one test
pytest --cov=p2p_portfolio                                           # coverage report
```

In [2]:
import subprocess
result = subprocess.run(["pytest", "tests/", "-v"], capture_output=True, text=True, cwd="..")
print(result.stdout[-2500:])

a.py::test_trade_log_round_trip PASSED                     [ 32%]
tests/test_data.py::test_trade_log_aggregates PASSED                     [ 35%]
tests/test_forecasting.py::test_forecaster_uses_heuristic_below_ml_threshold PASSED [ 38%]
tests/test_forecasting.py::test_forecaster_switches_to_model_above_threshold PASSED [ 41%]
tests/test_forecasting.py::test_forecaster_with_no_data_returns_default PASSED [ 45%]
tests/test_forecasting.py::test_best_windows_shape PASSED                [ 48%]
tests/test_metrics_compounding.py::test_turnover_ratio PASSED            [ 51%]
tests/test_metrics_compounding.py::test_weekly_return_on_capital_matches_worked_example PASSED [ 54%]
tests/test_metrics_compounding.py::test_payback_flat_withdrawal_matches_worked_example PASSED [ 58%]
tests/test_metrics_compounding.py::test_payback_full_reinvestment_matches_worked_example PASSED [ 61%]
tests/test_metrics_compounding.py::test_full_reinvestment_recovers_capital_faster_than_flat_withdrawal PASSED [ 64%]
tes

## 3. What pytest actually does

pytest is a test **runner**: it finds functions named `test_*` inside files named `test_*.py`, executes each one, and reports whether its `assert` statements held.

A minimal example — not part of the real suite, just to see the mechanics up close:

In [3]:
def captured_spread_pct(your_rate, reference_rate):
    return (your_rate - reference_rate) / reference_rate

# This is exactly the shape of a pytest test function.
# pytest discovers it by name (test_*), runs it, and reports pass/fail.
def test_demo_captured_spread():
    result = captured_spread_pct(1020, 1000)
    assert abs(result - 0.02) < 1e-9, f"expected 0.02, got {result}"

test_demo_captured_spread()
print("Passed — in a real pytest run this would show as 'PASSED' with -v, with no need to call it manually.")

Passed — in a real pytest run this would show as 'PASSED' with -v, with no need to call it manually.


**Why `pytest.approx(...)` instead of `==` for floats**: floating point arithmetic accumulates tiny rounding errors (`0.019999999999998` vs `0.02`), so a strict `==` comparison can fail even when the math is correct. `pytest.approx` allows a small tolerance — this is what our real tests use throughout `tests/`.

**Why this matters for this project specifically**: every formula we hand-calculated together (₦17.9M required volume, 5.95% weekly return, 16.8-week payback) is locked in as a test. A refactor that silently breaks the math gets caught in under 2 seconds, before it ships as a wrong number feeding a real financial decision.

## 4. Confirming the editable install actually works

The real test: does `import p2p_portfolio` work from a directory that has nothing to do with the project?

In [4]:
import subprocess

# Run from /tmp — nowhere near the project source — to prove it's not a local-folder fluke
check = subprocess.run(
    ["python3", "-c", "import p2p_portfolio; print(p2p_portfolio.__file__); print(p2p_portfolio.__version__)"],
    capture_output=True, text=True, cwd="/tmp",
)
print(check.stdout)
print(check.stderr)

/home/claude/p2p-portfolio/src/p2p_portfolio/__init__.py
0.1.0




In [5]:
# pip's own record of the editable install
pip_show = subprocess.run(["pip", "show", "p2p-portfolio"], capture_output=True, text=True)
print(pip_show.stdout)

Name: p2p-portfolio
Version: 0.1.0
Summary: Portfolio management and forecasting for a P2P USDT/naira trading operation
Home-page: 
Author: Dubem
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Editable project location: /home/claude/p2p-portfolio
Requires: numpy, pandas, scikit-learn
Required-by: 



The `__file__` path should point back at your real `src/p2p_portfolio/__init__.py`, and `pip show` should include an `Editable project location:` line — that line only appears for editable installs, confirming pip is linking to your live source rather than a frozen copy.

## 5. How TOML is written

TOML's building blocks:

| Syntax | Meaning | Example |
|---|---|---|
| `[section]` | a table (like a dict) | `[project]` |
| `[section.subsection]` | a nested table | `[tool.pytest.ini_options]` |
| `key = "value"` | string | `name = "p2p-portfolio"` |
| `key = 123` | number | `line-length = 100` |
| `key = true` | boolean | `required = true` |
| `key = ["a", "b"]` | array | `ignore = ["DTZ001"]` |
| `[[section]]` | array of tables (repeatable blocks) | not used in this project |
| `# comment` | comment | anything after `#` |

No meaningful indentation (unlike YAML) — what matters is correct nesting and quoting.

Let's parse this project's real `pyproject.toml` with Python's built-in `tomllib` (3.11+) to see TOML turn into a plain dict:

In [6]:
import tomllib

with open("../pyproject.toml", "rb") as f:
    config = tomllib.load(f)

print("Project name:   ", config["project"]["name"])
print("Version:        ", config["project"]["version"])
print("Dependencies:   ", config["project"]["dependencies"])
print("Dev extras:     ", config["project"]["optional-dependencies"]["dev"])
print("Console script: ", config["project"]["scripts"])

Project name:    p2p-portfolio
Version:         0.1.0
Dependencies:    ['pandas>=2.0', 'numpy>=1.24', 'scikit-learn>=1.3']
Dev extras:      ['pytest>=7.0', 'pytest-cov>=4.0', 'ruff>=0.4']
Console script:  {'p2p-portfolio': 'p2p_portfolio.__main__:main'}


That `optional-dependencies"]["dev"]` lookup is exactly what `pip install -e ".[dev]"`'s `[dev]` refers to — the bracket syntax in the pip command maps directly onto this TOML structure.

## 6. Where `pip show` gets its information

`pip show` does **not** re-read `pyproject.toml`. At install time, the build backend (`setuptools`, declared in `[build-system]`) translates `pyproject.toml`'s `[project]` table into a standardized `METADATA` file, written into a `*.dist-info/` folder. `pip show` just reads that pre-generated file — it never touches your TOML again after install.

In [7]:
import pathlib

dist_info = pathlib.Path("/usr/local/lib/python3.12/dist-packages/p2p_portfolio-0.1.0.dist-info")
metadata = (dist_info / "METADATA").read_text()
print(metadata)

print("---")
print("Other files generated alongside METADATA:")
for f in sorted(dist_info.iterdir()):
    print(" ", f.name)

Metadata-Version: 2.4
Name: p2p-portfolio
Version: 0.1.0
Summary: Portfolio management and forecasting for a P2P USDT/naira trading operation
Author: Dubem
License: MIT
Requires-Python: >=3.10
Description-Content-Type: text/markdown
Requires-Dist: pandas>=2.0
Requires-Dist: numpy>=1.24
Requires-Dist: scikit-learn>=1.3
Provides-Extra: dev
Requires-Dist: pytest>=7.0; extra == "dev"
Requires-Dist: pytest-cov>=4.0; extra == "dev"
Requires-Dist: ruff>=0.4; extra == "dev"

---
Other files generated alongside METADATA:
  INSTALLER
  METADATA
  RECORD
  REQUESTED
  WHEEL
  direct_url.json
  entry_points.txt
  top_level.txt


Quick reference for what each generated file is for:

| File | What it holds |
|---|---|
| `METADATA` | what `pip show` displays |
| `RECORD` | every installed file + checksum — used by `pip uninstall` |
| `entry_points.txt` | the `p2p-portfolio` console-script mapping |
| `direct_url.json` | records this came from a local editable install, and its path |

**The chain, in short**: `pyproject.toml` is the source of truth *you* write. `METADATA` is a derived snapshot generated once, at install time. If you change `version` in `pyproject.toml` later, `pip show` won't reflect it until you run `pip install -e .` again.

## Summary

| Question | One-line answer |
|---|---|
| How do I run the tests? | `pytest tests/ -v` from the project root, after `pip install -e ".[dev]"` |
| What does `pip install -e .` do? | Links your environment to live source instead of copying it — edits take effect immediately |
| What is pytest? | A test runner: discovers `test_*` functions, runs their asserts, reports pass/fail |
| How is TOML written? | Tables (`[section]`), key-value pairs, and arrays — no meaningful indentation |
| Where does `pip show` get its data? | A `METADATA` file generated from `pyproject.toml` at install time, not read live |
